# Data Cleaning

The goal of this notebook is to clean the dataset and ensure it is
consistent with the assumptions required for a valid A/B test analysis.

In particular, we will:
- Check for duplicate users
- Validate group assignments
- Remove inconsistent rows
- Prepare the dataset for analysis

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/AB Testing Data.csv")
df.head()

## Initial Inspection

We start by inspecting the dataset structure, including number of rows,
columns, and basic information.

In [ ]:
df.shape

In [ ]:
df.info()

The dataset contains a large number of observations and appears to be well-structured, with no immediately obvious missing values. The only point to pay attention is the fact that the timestamp is stored as string.

Besides, it contains additional user-level features (e.g., age, device, location),
which suggests it may not correspond to a minimal A/B testing setup.
These variables may be useful in later segmentation and validation steps.

## Missing Values

Missing values can bias the analysis or indicate data collection issues.
We verify whether any column contains null values.

In [ ]:
missing_values = df.isna().sum()
missing_values

No missing values were found across the dataset. This suggests that data
collection was complete for all recorded variables.

# Type Verification

As shown in the initial inspection, the timestamp is the only column with an inappropriate type, as it is stored as a string.

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [ ]:
df.info()

The timestamp column was converted to datetime format for potential
temporal analysis in later stages.

## Checking for Duplicate Users

Each user should appear only once in the dataset. Duplicate user entries
could bias the results and violate the assumption of independent samples.

In [ ]:
duplicate_summary = pd.Series({
    "duplicate_user_ids": df["user_id"].duplicated().sum(),
    "is_unique": df["user_id"].nunique() == len(df)
})

duplicate_summary

In [ ]:
assert duplicate_summary.duplicate_user_ids == 0
assert duplicate_summary.is_unique

No duplicate user_ids were found (0 duplicated entries). This confirms that each observation corresponds to a unique user, which is consistent with the assumptions of independent samples in A/B testing.

## Validating Group Assignments

Users assigned to the control group should only see the old page, and users in the treatment group should only see the new page.

We check for inconsistencies between group assignment and page exposure.

In [ ]:
mismatch = (
    ((df["group"] == "control") & (df["landing_page"] == "new_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "old_page"))
)

mismatch.sum()

In [ ]:
assert mismatch.sum() == 0, "Group assignment mismatch detected"

No mismatch was found. As an additional validation step, we verify the consistency by checking
the complementary condition.

In [ ]:
match = (
    ((df["group"] == "control") & (df["landing_page"] == "old_page")) |
    ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
)

match.sum() == len(df)

In [ ]:
assert match.sum() == len(df), "Group/page consistency validation failed"

Again, no inconsistencies were found between group assignment and page exposure.
While this is ideal, it is somewhat uncommon in real-world data and may
indicate that the dataset was pre-cleaned or synthetically generated.

## Group Distribution

We verify that users are approximately evenly split between control and
treatment groups.

In [ ]:
df["group"].value_counts(normalize=True)

The dataset shows an approximately balanced distribution between groups,
which is expected in a properly randomized A/B test. A formal validation of the allocation will be performed in the next stage.

## Final Dataset Overview

The dataset has been validated. It satisfies the key assumptions
required for A/B testing analysis:

- No missing values
- No duplicate users
- Consistent group assignments
- Balanced group distribution

The data is now ready for experiment validation and statistical analysis.

## Saving Cleaned Dataset

In [ ]:
df.to_csv("../data/processed/ab_data_cleaned.csv", index=False)